# 📘 Session 9: Advanced Pandas — DataFrame Operations & Data Manipulation
### Duration: ~2 Hours

---

**Topics Covered:**
1. Advanced DataFrame Indexing & Selection
2. DataFrame Operations & Transformations
3. Handling Missing Data
4. Data Aggregation & Grouping
5. Merging & Joining DataFrames
6. Reshaping & Pivoting Data

---

**Why This Session Matters for Data Science:**
- **Data manipulation** is 80% of data science work
- **Real datasets** require cleaning, filtering, combining
- **Feature engineering** happens through DataFrame operations
- **Data preparation** for ML models requires these skills
- **Performance** matters — efficient operations on large datasets

> Building on Session 8's Pandas basics, this session focuses on practical data manipulation techniques used daily in data science.

---
## 1. Review: Pandas Basics (from Session 8)

Quick review of core concepts before diving deeper:

In [ ]:
# --- Quick Pandas Review ---
import pandas as pd
import numpy as np

# Set display options
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 8)

# Create sample DataFrame
df = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'Age': [25, 30, 35, 28],
    'Department': ['Engineering', 'Marketing', 'Engineering', 'Sales'],
    'Salary': [75000, 85000, 95000, 78000]
})

print("Sample Employee DataFrame:")
print(df)
print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Index: {list(df.index)}")

---
## 2. Advanced DataFrame Indexing & Selection

Beyond basic `df['column']` and `df.loc[]`, pandas offers powerful selection methods:

### Multi-Level Indexing

| Method | Use Case | Example |
|--------|----------|---------|
| `.loc[row, col]` | Label-based selection | `df.loc[0, 'Name']` |
| `.iloc[row, col]` | Position-based selection | `df.iloc[0, 1]` |
| `.at[row, col]` | Fast scalar access by label | `df.at[0, 'Name']` |
| `.iat[row, col]` | Fast scalar access by position | `df.iat[0, 1]` |

### Boolean & Conditional Selection

| Pattern | Description | Example |
|---------|-------------|---------|
| `df[condition]` | Single condition | `df[df['Age'] > 30]` |
| `df[(cond1) & (cond2)]` | Multiple AND conditions | `df[(df['Age'] > 25) & (df['Department'] == 'Engineering')]` |
| `df[(cond1) | (cond2)]` | Multiple OR conditions | `df[(df['Salary'] > 80000) | (df['Years_Exp'] > 5)]` |
| `df[~condition]` | Negation (NOT) | `df[~df['Department'].isin(['HR', 'Sales'])]` |
| `.isin(values)` | Check membership | `df[df['Department'].isin(['Engineering', 'Marketing'])]` |
| `.between(a, b)` | Range check | `df[df['Age'].between(25, 35)]` |

### String Pattern Matching

| Method | Description | Example |
|--------|-------------|---------|
| `.str.contains(pattern)` | Contains substring | `df[df['Name'].str.contains('A')]` |
| `.str.startswith(prefix)` | Starts with | `df[df['Department'].str.startswith('Eng')]` |
| `.str.endswith(suffix)` | Ends with | `df[df['Name'].str.endswith('e')]` |
| `.str.match(regex)` | Regex match | `df[df['Name'].str.match(r'^A.*e$')]` |

> ⚠️ **Special Case — Boolean indexing performance**: Use `&` and `|` instead of `and`/`or` for element-wise operations. Python's `and`/`or` don't work with pandas boolean arrays.

> ⚠️ **Special Case — `.at` vs `.loc`**: Use `.at` for single scalar access (faster), `.loc` for slices or multiple values.

In [ ]:
# --- Advanced Indexing Examples ---

employees = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'Age': [25, 30, 35, 28, 32],
    'Department': ['Engineering', 'Marketing', 'Engineering', 'Sales', 'HR'],
    'Salary': [75000, 85000, 95000, 78000, 72000],
    'Years_Exp': [2, 5, 8, 3, 6]
})

print("Employee DataFrame:")
print(employees)
print()

# Complex boolean indexing
senior_engineers = employees[
    (employees['Department'] == 'Engineering') & 
    (employees['Years_Exp'] > 5) & 
    (employees['Salary'] > 90000)
]
print("Senior Engineers (Engineering + >5 years exp + >$90K):")
print(senior_engineers)
print()

# Using .isin() for multiple values
tech_departments = employees[employees['Department'].isin(['Engineering', 'Marketing'])]
print("Tech Departments (Engineering or Marketing):")
print(tech_departments)
print()

# String pattern matching
names_with_a = employees[employees['Name'].str.contains('a', case=False)]
print("Names containing 'a' (case insensitive):")
print(names_with_a)
print()

# Age range
mid_career = employees[employees['Age'].between(28, 40)]
print("Mid-career employees (28-40 years old):")
print(mid_career)

In [ ]:
# --- Advanced Selection Methods ---

# .loc with multiple conditions
result = employees.loc[
    (employees['Department'] == 'Engineering') & (employees['Age'] < 30),
    ['Name', 'Salary', 'Years_Exp']
]
print("Engineering employees under 30 - Name, Salary, Experience:")
print(result)
print()

# Using .query() for readable conditions
query_result = employees.query('Department == "Engineering" and Years_Exp >= 5')
print("Engineers with 5+ years experience (using .query()):")
print(query_result)
print()

# Fast scalar access
print("Fast scalar access:")
print(f"  df.at[0, 'Name']: {employees.at[0, 'Name']}")
print(f"  df.iat[0, 1]: {employees.iat[0, 1]} (Age column)")
print()

# Selecting with callable
numeric_cols = employees.loc[:, lambda df: df.select_dtypes(include='number').columns]
print("Only numeric columns:")
print(numeric_cols.head())

---
## 3. DataFrame Operations & Transformations

Powerful operations for data manipulation:

### Column Operations

| Operation | Method | Example |
|-----------|--------|---------|
| Add column | `df['new'] = expression` | `df['Bonus'] = df['Salary'] * 0.1` |
| Transform column | `df['col'] = df['col'].apply(func)` | `df['Name'] = df['Name'].str.upper()` |
| Rename columns | `df.rename(columns=dict)` | `df.rename(columns={'old': 'new'})` |
| Drop columns | `df.drop(columns=list)` | `df.drop(columns=['temp'])` |
| Reorder columns | `df[desired_order]` | `df[['A', 'B', 'C']]` |

### Row Operations

| Operation | Method | Example |
|-----------|--------|---------|
| Add row | `df.loc[len(df)] = values` | `df.loc[len(df)] = ['New', 30, ...]` |
| Drop rows | `df.drop(index=list)` | `df.drop(index=[0, 2])` |
| Filter rows | `df[condition]` | `df[df['Age'] > 30]` |
| Sort rows | `df.sort_values(by=col)` | `df.sort_values(by='Salary', ascending=False)` |
| Reset index | `df.reset_index()` | `df.reset_index(drop=True)` |

### Apply, Map, ApplyMap

| Method | Applies To | Returns | Use Case |
|--------|------------|---------|----------|
| `.apply(func)` | Series or DataFrame rows/columns | Series or DataFrame | Row/column-wise operations |
| `.map(func)` | Series elements | Series | Element-wise operations on Series |
| `.applymap(func)` | DataFrame elements | DataFrame | Element-wise operations on DataFrame |

> ⚠️ **Special Case — `.apply()` vs `.map()`**: Use `.map()` for Series element-wise operations, `.apply()` for more complex row/column operations.

> ⚠️ **Special Case — Vectorized operations**: Prefer `df['A'] * 2` over `df['A'].apply(lambda x: x * 2)` for performance.

In [ ]:
# --- Column Operations ---

employees = pd.DataFrame({
    'Name': ['alice', 'bob', 'charlie', 'diana'],
    'Age': [25, 30, 35, 28],
    'Salary': [75000, 85000, 95000, 78000],
    'Department': ['eng', 'mkt', 'eng', 'sales']
})

print("Original DataFrame:")
print(employees)
print()

# Add calculated columns
employees['Bonus'] = employees['Salary'] * 0.1
employees['Total_Comp'] = employees['Salary'] + employees['Bonus']
employees['Salary_per_Age'] = employees['Salary'] / employees['Age']

print("After adding calculated columns:")
print(employees)
print()

# Transform existing columns
employees['Name'] = employees['Name'].str.title()
employees['Department'] = employees['Department'].str.upper()

print("After transforming Name and Department:")
print(employees)
print()

# Rename columns
employees = employees.rename(columns={
    'Salary_per_Age': 'Salary_to_Age_Ratio',
    'Total_Comp': 'Total_Compensation'
})

print("After renaming columns:")
print(employees)

In [ ]:
# --- Apply, Map, ApplyMap ---

employees = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Age': [25, 30, 35],
    'Salary': [75000, 85000, 95000],
    'Department': ['Engineering', 'Marketing', 'Engineering']
})

# .map() for Series element-wise operations
def get_experience_level(age):
    if age < 28:
        return 'Junior'
    elif age < 35:
        return 'Mid'
    else:
        return 'Senior'

employees['Experience_Level'] = employees['Age'].map(get_experience_level)
print("Using .map() for experience level:")
print(employees[['Name', 'Age', 'Experience_Level']])
print()

# .apply() for Series operations
employees['Salary_Category'] = employees['Salary'].apply(
    lambda x: 'High' if x > 90000 else 'Medium' if x > 80000 else 'Low'
)
print("Using .apply() for salary categories:")
print(employees[['Name', 'Salary', 'Salary_Category']])
print()

# .apply() for DataFrame row operations
def calculate_performance_score(row):
    base_score = row['Salary'] / 1000
    age_bonus = row['Age'] / 10
    return base_score + age_bonus

employees['Performance_Score'] = employees.apply(calculate_performance_score, axis=1)
print("Using .apply(axis=1) for row-wise calculations:")
print(employees[['Name', 'Performance_Score']])
print()

# .applymap() for element-wise DataFrame operations
numeric_df = employees.select_dtypes(include='number')
normalized_df = numeric_df.applymap(lambda x: x / numeric_df.max().max())
print("Using .applymap() for normalization (first 3 rows):")
print(normalized_df.head(3))

---
## 4. Handling Missing Data

Real-world data is messy — missing values are common:

### Detecting Missing Data

| Method | Description | Example |
|--------|-------------|---------|
| `.isnull()` | True for NaN/None | `df.isnull()` |
| `.notnull()` | True for valid values | `df.notnull()` |
| `.isna()` | Same as isnull() | `df.isna()` |
| `.info()` | Summary with non-null counts | `df.info()` |

### Handling Missing Data

| Strategy | Method | Use When |
|----------|--------|----------|
| **Drop** | `df.dropna()` | Few missing values, random |
| **Fill with value** | `df.fillna(value)` | Meaningful default exists |
| **Fill with mean/median** | `df.fillna(df.mean())` | Numeric data, normal distribution |
| **Forward fill** | `df.fillna(method='ffill')` | Time series, carry forward |
| **Backward fill** | `df.fillna(method='bfill')` | Time series, carry backward |
| **Interpolate** | `df.interpolate()` | Time series, smooth transitions |

### Advanced Missing Data Handling

| Method | Description | Example |
|--------|-------------|---------|
| `thresh` parameter | Drop if fewer than n non-null | `df.dropna(thresh=3)` |
| `subset` parameter | Only consider certain columns | `df.dropna(subset=['A', 'B'])` |
| `inplace=True` | Modify in place | `df.dropna(inplace=True)` |

> ⚠️ **Special Case — Data loss**: `dropna()` can remove too much data. Always check how many rows will be dropped first.

> ⚠️ **Special Case — Fill strategy**: Choose fill method based on data meaning. Mean for normal data, median for skewed data, mode for categorical.

> **Data Science relevance**: Missing data handling affects model accuracy. Poor handling can introduce bias or reduce sample size.

In [ ]:
# --- Missing Data Examples ---

# Create DataFrame with missing data
sales_data = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'],
    'Sales': [1000, np.nan, 1200, np.nan, 1400, 1300],
    'Customers': [50, 45, np.nan, 55, 60, np.nan],
    'Region': ['North', 'South', None, 'North', 'South', 'North']
})

print("DataFrame with missing values:")
print(sales_data)
print()

# Detect missing data
print("Missing data detection:")
print(f"  Is null:\n{sales_data.isnull()}")
print(f"  Sum of nulls per column:\n{sales_data.isnull().sum()}")
print(f"  Total nulls: {sales_data.isnull().sum().sum()}")
print()

# Different fill strategies
print("Fill strategies:")

# Fill with specific value
filled_zero = sales_data.fillna(0)
print("Filled with 0:")
print(filled_zero)
print()

# Fill numeric with mean
numeric_cols = sales_data.select_dtypes(include='number').columns
filled_mean = sales_data.copy()
filled_mean[numeric_cols] = filled_mean[numeric_cols].fillna(filled_mean[numeric_cols].mean())
print("Numeric columns filled with mean:")
print(filled_mean)
print()

# Forward fill
filled_ffill = sales_data.fillna(method='ffill')
print("Forward fill:")
print(filled_ffill)
print()

# Fill categorical with mode
filled_mode = sales_data.copy()
filled_mode['Region'] = filled_mode['Region'].fillna(filled_mode['Region'].mode()[0])
print("Region filled with mode:")
print(filled_mode)

In [ ]:
# --- Advanced Missing Data Handling ---

messy_data = pd.DataFrame({
    'A': [1, 2, np.nan, 4, 5],
    'B': [np.nan, 2, 3, np.nan, 5],
    'C': [1, np.nan, np.nan, 4, 5],
    'D': ['a', 'b', 'c', 'd', 'e']
})

print("Messy DataFrame:")
print(messy_data)
print()

# Drop rows with any missing values
drop_any = messy_data.dropna()
print("Drop rows with ANY missing values:")
print(drop_any)
print()

# Drop rows with all missing values
drop_all = messy_data.dropna(how='all')
print("Drop rows with ALL missing values:")
print(drop_all)
print()

# Drop with threshold (keep rows with at least 3 non-null values)
drop_thresh = messy_data.dropna(thresh=3)
print("Drop rows with fewer than 3 non-null values:")
print(drop_thresh)
print()

# Drop only considering specific columns
drop_subset = messy_data.dropna(subset=['A', 'B'])
print("Drop rows where A or B is missing:")
print(drop_subset)
print()

# Interpolate numeric data
interpolated = messy_data.select_dtypes(include='number').interpolate()
print("Interpolated numeric columns:")
print(interpolated)

---
## 5. Data Aggregation & Grouping

Group data and compute aggregate statistics:

### GroupBy Operations

| Method | Description | Example |
|--------|-------------|---------|
| `.groupby(col)` | Group by column | `df.groupby('Department')` |
| `.agg(func)` | Apply aggregation | `.agg({'Salary': 'mean'})` |
| `.size()` | Count per group | `df.groupby('Dept').size()` |
| `.count()` | Non-null count per group | `df.groupby('Dept').count()` |
| `.mean()`, `.sum()`, etc. | Built-in aggregations | `df.groupby('Dept').mean()` |

### Multiple Grouping

| Pattern | Description | Example |
|---------|-------------|---------|
| Single group | One column | `df.groupby('Department')` |
| Multiple groups | List of columns | `df.groupby(['Dept', 'City'])` |
| Hierarchical | MultiIndex result | `df.groupby(['A', 'B']).sum()` |

### Custom Aggregation

| Method | Use Case | Example |
|--------|----------|---------|
| `.agg(func)` | Single function | `.agg(np.mean)` |
| `.agg([f1, f2])` | Multiple functions | `.agg(['mean', 'std'])` |
| `.agg({'col': func})` | Different funcs per column | `.agg({'Salary': 'mean', 'Age': 'max'})` |
| `.apply(func)` | Custom function | `.apply(lambda x: x.max() - x.min())` |

> ⚠️ **Special Case — GroupBy memory**: GroupBy creates intermediate objects. For large datasets, consider chunked processing.

> ⚠️ **Special Case — Aggregation functions**: Built-in methods (`.mean()`, `.sum()`) are faster than `.agg('mean')`.

> **Data Science relevance**: GroupBy is essential for cohort analysis, A/B testing, segmentation, and feature engineering.

In [ ]:
# --- GroupBy Operations ---

employees = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'Department': ['Engineering', 'Marketing', 'Engineering', 'Sales', 'HR', 'Engineering'],
    'Salary': [75000, 85000, 95000, 78000, 72000, 105000],
    'Age': [25, 30, 35, 28, 32, 45],
    'City': ['NYC', 'LA', 'Chicago', 'Boston', 'Miami', 'Seattle']
})

print("Employee DataFrame:")
print(employees)
print()

# Basic groupby
dept_groups = employees.groupby('Department')
print("GroupBy object:")
print(f"  Groups: {list(dept_groups.groups.keys())}")
print(f"  Size per group: {dept_groups.size()}")
print()

# Aggregate by department
dept_stats = employees.groupby('Department').agg({
    'Salary': ['mean', 'max', 'min', 'count'],
    'Age': ['mean', 'max']
})
print("Department statistics:")
print(dept_stats)
print()

# Multiple grouping
city_dept = employees.groupby(['City', 'Department']).size()
print("Employees by City and Department:")
print(city_dept)
print()

# Custom aggregation with apply
def salary_range(group):
    return group['Salary'].max() - group['Salary'].min()

salary_spread = employees.groupby('Department').apply(salary_range)
print("Salary range (max - min) by department:")
print(salary_spread)

In [ ]:
# --- Advanced GroupBy ---

# Create more complex dataset
sales = pd.DataFrame({
    'Date': pd.date_range('2023-01-01', periods=12, freq='M'),
    'Product': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'A', 'B', 'C', 'A', 'B'],
    'Region': ['North', 'South', 'North', 'East', 'South', 'North', 'East', 'North', 'South', 'East', 'North', 'South'],
    'Sales': [1000, 1200, 900, 1100, 1300, 950, 1050, 1150, 1250, 1000, 1100, 1350],
    'Quantity': [10, 12, 9, 11, 13, 9.5, 10.5, 11.5, 12.5, 10, 11, 13.5]
})

print("Sales DataFrame:")
print(sales.head())
print()

# Multiple aggregations
product_stats = sales.groupby('Product').agg({
    'Sales': ['sum', 'mean', 'std'],
    'Quantity': ['sum', 'mean'],
    'Date': 'count'  # Number of transactions
})
print("Product performance:")
print(product_stats)
print()

# Group by multiple columns
region_product = sales.groupby(['Region', 'Product']).agg({
    'Sales': 'sum',
    'Quantity': 'mean'
})
print("Sales by Region and Product:")
print(region_product)
print()

# Using transform for group-wise operations
sales['Sales_Percentile'] = sales.groupby('Product')['Sales'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
print("Sales percentile within each product:")
print(sales[['Product', 'Sales', 'Sales_Percentile']].head())
print()

# Filter groups based on condition
big_regions = sales.groupby('Region').filter(lambda x: x['Sales'].sum() > 3000)
print("Only regions with total sales > 3000:")
print(big_regions[['Region', 'Sales']].head())

---
## 6. Merging & Joining DataFrames

Combine data from multiple DataFrames:

### Join Types

| Join Type | Description | SQL Equivalent | Use When |
|-----------|-------------|----------------|----------|
| **inner** | Only matching rows | `INNER JOIN` | Need complete data |
| **left** | All left rows + matches | `LEFT JOIN` | Keep all left data |
| **right** | All right rows + matches | `RIGHT JOIN` | Keep all right data |
| **outer** | All rows from both | `FULL OUTER JOIN` | Keep everything |

### Merge Methods

| Method | Syntax | Use Case |
|--------|--------|----------|
| `pd.merge()` | `pd.merge(left, right, on='key')` | Standard merge |
| `df.join()` | `left.join(right, on='key')` | Index-based join |
| `pd.concat()` | `pd.concat([df1, df2])` | Stack DataFrames |

### Merge Parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `on` | Column to join on | `on='customer_id'` |
| `left_on`, `right_on` | Different column names | `left_on='id', right_on='customer_id'` |
| `how` | Join type | `how='left'` |
| `suffixes` | Suffix for duplicate columns | `suffixes=('_left', '_right')` |

> ⚠️ **Special Case — Index joins**: Use `df.join()` when joining on index. Use `pd.merge()` when joining on columns.

> ⚠️ **Special Case — Many-to-many joins**: Can create Cartesian products. Check result size before proceeding.

> **Data Science relevance**: Real datasets come from multiple sources. Merging combines customer data, transaction data, product data, etc.

In [ ]:
# --- Merging DataFrames ---

# Customer data
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'city': ['NYC', 'LA', 'Chicago', 'Boston', 'Miami']
})

# Order data
orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106],
    'customer_id': [1, 2, 2, 3, 4, 6],  # Note: customer 6 doesn't exist
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Cable', 'Tablet'],
    'amount': [1200, 50, 100, 300, 25, 800]
})

print("Customers:")
print(customers)
print()
print("Orders:")
print(orders)
print()

# Inner join (only matching customers)
inner_join = pd.merge(customers, orders, on='customer_id', how='inner')
print("Inner join (only customers with orders):")
print(inner_join)
print()

# Left join (all customers, even without orders)
left_join = pd.merge(customers, orders, on='customer_id', how='left')
print("Left join (all customers):")
print(left_join)
print()

# Right join (all orders, even for non-existent customers)
right_join = pd.merge(customers, orders, on='customer_id', how='right')
print("Right join (all orders):")
print(right_join)
print()

# Outer join (everything)
outer_join = pd.merge(customers, orders, on='customer_id', how='outer')
print("Outer join (everything):")
print(outer_join)

In [ ]:
# --- Advanced Merging ---

# Employee data
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'dept_id': [10, 20, 10, 30]
})

# Department data
departments = pd.DataFrame({
    'dept_id': [10, 20, 30, 40],
    'dept_name': ['Engineering', 'Marketing', 'Sales', 'HR'],
    'budget': [500000, 300000, 400000, 200000]
})

# Salary data
salaries = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'salary': [75000, 85000, 95000, 78000, 72000],
    'bonus': [7500, 8500, 9500, 7800, 7200]
})

print("Employees:")
print(employees)
print()
print("Departments:")
print(departments)
print()
print("Salaries:")
print(salaries)
print()

# Merge employees with departments
emp_dept = pd.merge(employees, departments, on='dept_id', how='left')
print("Employees with department info:")
print(emp_dept)
print()

# Merge with salaries (different column names)
complete_data = pd.merge(emp_dept, salaries, left_on='emp_id', right_on='emp_id', how='outer')
print("Complete employee data:")
print(complete_data)
print()

# Using join() with index
dept_indexed = departments.set_index('dept_id')
emp_dept_join = employees.set_index('dept_id').join(dept_indexed, how='left')
print("Using join() with index:")
print(emp_dept_join.reset_index())

---
## 7. Reshaping & Pivoting Data

Transform data between wide and long formats:

### Reshaping Methods

| Method | Description | Use Case |
|--------|-------------|----------|
| `melt()` | Wide to long | Convert columns to rows |
| `pivot()` | Long to wide | Convert rows to columns |
| `pivot_table()` | Pivot with aggregation | Grouped pivot with stats |
| `stack()` | Columns to index | Multi-level columns to index |
| `unstack()` | Index to columns | Multi-level index to columns |

### Melt vs Pivot

| Operation | Input Format | Output Format | Method |
|-----------|---------------|----------------|--------|
| **Melt** | Wide (many columns) | Long (few columns) | `df.melt()` |
| **Pivot** | Long (few columns) | Wide (many columns) | `df.pivot()` |

### Pivot Table Features

| Feature | Description | Example |
|---------|-------------|---------|
| `values` | Column to aggregate | `values='Sales'` |
| `index` | Row grouping | `index='Region'` |
| `columns` | Column grouping | `columns='Product'` |
| `aggfunc` | Aggregation function | `aggfunc='sum'` |
| `margins` | Add totals | `margins=True` |

> ⚠️ **Special Case — Melt**: `melt()` makes data "tidy" for analysis. Each observation becomes one row.

> ⚠️ **Special Case — Pivot**: Requires unique index/column combinations. Use `pivot_table()` for duplicates.

> **Data Science relevance**: Visualization tools prefer different formats. Reshaping prepares data for plotting and modeling.

In [ ]:
# --- Melting Data (Wide to Long) ---

# Wide format data
grades_wide = pd.DataFrame({
    'Student': ['Alice', 'Bob', 'Charlie'],
    'Math': [85, 92, 78],
    'Science': [88, 95, 82],
    'English': [90, 87, 85]
})

print("Wide format (original):")
print(grades_wide)
print()

# Melt to long format
grades_long = grades_wide.melt(
    id_vars=['Student'],  # Columns to keep as-is
    value_vars=['Math', 'Science', 'English'],  # Columns to melt
    var_name='Subject',  # Name for variable column
    value_name='Grade'  # Name for value column
)

print("Long format (melted):")
print(grades_long)
print()

# Melt everything except Student
grades_auto = grades_wide.melt(id_vars=['Student'])
print("Auto-melted (everything except Student):")
print(grades_auto)

In [ ]:
# --- Pivoting Data (Long to Wide) ---

# Long format sales data
sales_long = pd.DataFrame({
    'Date': ['2023-01', '2023-01', '2023-02', '2023-02', '2023-03', '2023-03'],
    'Product': ['A', 'B', 'A', 'B', 'A', 'B'],
    'Sales': [1000, 1200, 1100, 1300, 1050, 1250]
})

print("Long format sales data:")
print(sales_long)
print()

# Pivot to wide format
sales_wide = sales_long.pivot(
    index='Date',      # Rows
    columns='Product', # Columns
    values='Sales'     # Values
)

print("Wide format (pivoted):")
print(sales_wide)
print()

# Reset index for cleaner display
print("With reset index:")
print(sales_wide.reset_index())

In [ ]:
# --- Pivot Tables ---

# Detailed sales data
detailed_sales = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South', 'North', 'South', 'North', 'South'],
    'Product': ['A', 'B', 'A', 'B', 'A', 'A', 'B', 'B'],
    'Month': ['Jan', 'Jan', 'Jan', 'Jan', 'Feb', 'Feb', 'Feb', 'Feb'],
    'Sales': [1000, 1200, 900, 1100, 1100, 950, 1300, 1150],
    'Quantity': [10, 12, 9, 11, 11, 9.5, 13, 11.5]
})

print("Detailed sales data:")
print(detailed_sales)
print()

# Pivot table with aggregation
sales_pivot = pd.pivot_table(
    detailed_sales,
    values='Sales',      # What to aggregate
    index='Region',      # Rows
    columns='Month',     # Columns
    aggfunc='sum',       # How to aggregate
    margins=True         # Add totals
)

print("Sales pivot table:")
print(sales_pivot)
print()

# Multiple aggregations
multi_pivot = pd.pivot_table(
    detailed_sales,
    values=['Sales', 'Quantity'],
    index='Region',
    columns='Product',
    aggfunc={'Sales': 'sum', 'Quantity': 'mean'}
)

print("Multiple aggregations pivot:")
print(multi_pivot)

---
## 📝 Session 9 Summary

### What You Learned

| Topic | Key Takeaway |
|-------|---------------|
| **Advanced Indexing** | `.loc[]`, `.iloc[]`, boolean masks, `.query()`, string patterns |
| **Data Operations** | Add/remove columns, `.apply()`, `.map()`, `.applymap()` |
| **Missing Data** | `.isnull()`, `.dropna()`, `.fillna()`, interpolation |
| **GroupBy** | `.groupby()`, `.agg()`, multiple groupings, custom functions |
| **Merging** | `pd.merge()`, join types (inner/left/right/outer), multi-table joins |
| **Reshaping** | `melt()` for wide→long, `pivot()` for long→wide, `pivot_table()` |

### Key Gotchas to Remember
- Boolean indexing uses `&` and `|` (not `and`/`or`)
- `df['col']` = Series; `df[['col']]` = DataFrame
- `loc` uses **labels**; `iloc` uses **integer positions**
- Vectorized operations (`df * 2`) > loops (`.apply()`)
- String methods need `.str.` prefix
- `dropna()` can remove too much data — check first
- GroupBy creates intermediate objects — memory intensive
- Many-to-many merges can create huge result sets
- `melt()` makes data "tidy"; `pivot()` makes it "wide"
- Use `pivot_table()` when pivot would have duplicates

### Data Science Connection
- **Data cleaning** = 60-80% of data science time
- **Feature engineering** = creating new columns from existing data
- **Grouping** = cohort analysis, A/B testing, segmentation
- **Merging** = combining datasets from multiple sources
- **Reshaping** = preparing data for visualization and modeling
- **Missing data** = affects model accuracy and bias

### Next Session
**Session 10**: Pandas Filtering, Sorting, and Advanced Operations — deep dive into data manipulation techniques.